# Phase 4 — Analyse des Résultats


In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

sys.path.insert(0, "modèle llm")

DATA_DIR   = Path("data")
MODELS_DIR = Path("models")

def load_pkl(name):
    with open(DATA_DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

X_train = load_pkl("X_train")
X_val   = load_pkl("X_val")
X_test  = load_pkl("X_test")
y_train = load_pkl("y_train")
y_val   = load_pkl("y_val")
y_test  = load_pkl("y_test")

print(f"X_train : {X_train.shape}  |  X_val : {X_val.shape}  |  X_test : {X_test.shape}")
print(f"Splits  : train ≤ {X_train.index.max().date()}  |  val ≤ {X_val.index.max().date()}  |  test ≤ {X_test.index.max().date()}")


In [ ]:
try:
    _ = dl_results
    _ = sklearn_val_results
    _ = results_df
    _ = best_name
    print("Variables de 03_models.ipynb détectées.")
except NameError:
    raise RuntimeError(
        "Les variables dl_results / sklearn_val_results sont introuvables.\n"
        "Exécutez 03_models.ipynb AVANT ce notebook (même session Jupyter / noyau partagé)."
    )

from config import SEQUENCE_LENGTH
from xgboost_model import XGBoostWrapper

SEQ_LEN = SEQUENCE_LENGTH

def make_sequences(X, y, seq_len):
    X_arr, y_arr = X.values, y.values
    Xs, ys = [], []
    for i in range(seq_len, len(X_arr)):
        Xs.append(X_arr[i - seq_len:i])
        ys.append(y_arr[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_vl_seq, y_vl_seq = make_sequences(X_val,  y_val,  SEQ_LEN)
X_te_seq, y_te_seq = make_sequences(X_test, y_test, SEQ_LEN)

# Probabilités val de chaque modèle DL
dl_val_probs = {}
for name, res in dl_results.items():
    dl_val_probs[name] = res["trainer"].predict(X_vl_seq)

# Probabilités val des modèles sklearn (nécessitent predict_proba)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

sklearn_models_trained = {
    "Dummy (most_frequent)": DummyClassifier(strategy="most_frequent").fit(X_train, y_train),
    "Logistic Regression":   LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train),
    "Random Forest":         RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train),
    "Gradient Boosting":     GradientBoostingClassifier(n_estimators=300, random_state=42).fit(X_train, y_train),
}

sklearn_val_probs = {}
for name, clf in sklearn_models_trained.items():
    if hasattr(clf, "predict_proba"):
        sklearn_val_probs[name] = clf.predict_proba(X_val)[:, 1]
    else:
        sklearn_val_probs[name] = clf.predict(X_val).astype(float)

print("Probabilités reconstruites pour", len(dl_val_probs) + len(sklearn_val_probs), "modèles.")


## 1 — Courbes ROC comparatives (validation set)

Chaque courbe représente le compromis entre TPR (sensibilité) et FPR (1-spécificité) pour un modèle.  
L'AUC-ROC synthétise la discrimination globale : 0.5 = aléatoire, 1.0 = parfait.  
On compare tous les modèles sur le même val set (2023) pour une sélection équitable.

In [ ]:
from sklearn.metrics import roc_curve, auc

all_val_probs = {}
for name, probs in sklearn_val_probs.items():
    all_val_probs[name] = {"probs": probs, "y_true": y_val.values}

for name, probs in dl_val_probs.items():
    all_val_probs[name] = {"probs": probs, "y_true": y_vl_seq}

# Palettes : sklearn en bleus, DL en chauds
sklearn_names = list(sklearn_val_probs.keys())
dl_names      = list(dl_val_probs.keys())
colors_sk = plt.cm.Blues(np.linspace(0.4, 0.9, len(sklearn_names)))
colors_dl = plt.cm.Oranges(np.linspace(0.4, 0.9, len(dl_names)))

fig, ax = plt.subplots(figsize=(10, 7))

for i, name in enumerate(sklearn_names):
    d = all_val_probs[name]
    fpr, tpr, _ = roc_curve(d["y_true"], d["probs"])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors_sk[i], lw=1.8,
            label=f"{name}  (AUC={roc_auc:.3f})", linestyle="--")

for i, name in enumerate(dl_names):
    d = all_val_probs[name]
    fpr, tpr, _ = roc_curve(d["y_true"], d["probs"])
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if name == best_name else 1.8
    ax.plot(fpr, tpr, color=colors_dl[i], lw=lw,
            label=f"{name}  (AUC={roc_auc:.3f})",
            linestyle="-" if name == best_name else "-.")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Aléatoire (AUC=0.500)")
ax.set_xlabel("Taux de faux positifs (FPR)", fontsize=12)
ax.set_ylabel("Taux de vrais positifs (TPR)", fontsize=12)
ax.set_title("Courbes ROC — tous les modèles (val set 2023)", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("models/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"→ Figure sauvegardée : models/roc_curves.png")


## 2 — Matrice de confusion du meilleur modèle (test set)

La matrice de confusion décompose les erreurs :  
- **VP** (vrai positif) : prédit hausse, BTC monte  
- **FP** (faux positif) : prédit hausse, BTC baisse → coût : position longue perdante  
- **FN** (faux négatif) : prédit baisse, BTC monte → coût : opportunité manquée  
- **VN** (vrai négatif) : prédit baisse, BTC baisse

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, roc_auc_score

# Probabilités test du meilleur modèle
best_trainer = dl_results[best_name]["trainer"]
y_prob_test  = best_trainer.predict(X_te_seq)
y_pred_test  = (y_prob_test >= 0.5).astype(int)

cm = confusion_matrix(y_te_seq, y_pred_test)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Prédit Baisse", "Prédit Hausse"],
            yticklabels=["Réel Baisse", "Réel Hausse"],
            ax=axes[0], linewidths=0.5, cbar=False,
            annot_kws={"size": 18, "weight": "bold"})
axes[0].set_title(f"Matrice de confusion — {best_name}\n(test set ≥ 2024)", fontsize=12)
axes[0].set_ylabel("Étiquette réelle", fontsize=11)
axes[0].set_xlabel("Prédiction", fontsize=11)

# Annotations manuelles (TP/FP/TN/FN)
for text, (r, c), color in [("VN", (0,0), "navy"), ("FP", (0,1), "darkred"),
                              ("FN", (1,0), "darkred"), ("VP", (1,1), "navy")]:
    axes[0].text(c + 0.5, r + 0.75, text, ha="center", va="center",
                 fontsize=11, color=color, style="italic")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Blues",
            xticklabels=["Prédit Baisse", "Prédit Hausse"],
            yticklabels=["Réel Baisse", "Réel Hausse"],
            ax=axes[1], linewidths=0.5, cbar=True,
            annot_kws={"size": 16})
axes[1].set_title(f"Matrice normalisée — {best_name}", fontsize=12)
axes[1].set_xlabel("Prédiction", fontsize=11)

plt.tight_layout()
plt.savefig("models/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'='*50}")
print(f"  {best_name} — TEST SET")
print(f"{'='*50}")
print(f"  F1        : {f1_score(y_te_seq, y_pred_test):.4f}")
print(f"  Accuracy  : {accuracy_score(y_te_seq, y_pred_test):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(y_te_seq, y_prob_test):.4f}")
print(f"  Précision : {tp/(tp+fp):.4f}  |  Rappel : {tp/(tp+fn):.4f}")
print(f"  Spécificité (TNR) : {tn/(tn+fp):.4f}")
print(f"\n{classification_report(y_te_seq, y_pred_test, target_names=['Baisse','Hausse'])}")


## 3 — Analyse des erreurs

On étudie **quand** le modèle se trompe : contexte de marché, amplitude des mouvements, régimes de volatilité.  
L'objectif est de comprendre si les erreurs sont structurelles (biais systématique) ou aléatoires (bruit de marché).

In [ ]:
# Les séquences commencent à l'index SEQ_LEN du test set
test_dates = X_test.index[SEQ_LEN:]

err_df = pd.DataFrame({
    "date":       test_dates,
    "y_true":     y_te_seq.astype(int),
    "y_pred":     y_pred_test,
    "y_prob":     y_prob_test,
    "correct":    (y_pred_test == y_te_seq.astype(int)),
    "error_type": np.where(
        y_pred_test == y_te_seq.astype(int), "correct",
        np.where(y_pred_test == 1, "FP (prédit hausse, baisse réelle)",
                                   "FN (prédit baisse, hausse réelle)")
    ),
}).set_index("date")

# Ajouter le rendement BTC réel (depuis X_test — colonne ret_1d_btc décalée donc = J-1)
# On charge le fichier brut pour le rendement non-décalé
try:
    merged = pd.read_csv("data/merged_daily.csv", index_col=0, parse_dates=True)
    btc_ret = merged["close_btc"].pct_change().rename("btc_ret_real")
    err_df = err_df.join(btc_ret, how="left")
    has_ret = True
except Exception:
    has_ret = False
    print("merged_daily.csv introuvable — analyse des rendements désactivée.")

print(f"Test set : {len(err_df)} jours")
print(f"Corrects : {err_df['correct'].sum()} ({err_df['correct'].mean():.1%})")
print(f"Erreurs  : {(~err_df['correct']).sum()}")
print(f"\nDécomposition des erreurs :")
print(err_df["error_type"].value_counts())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

bins = np.linspace(0, 1, 21)
for etype, color, label in [
    ("correct",                          "steelblue", "Correct"),
    ("FP (prédit hausse, baisse réelle)", "tomato",    "Faux Positif"),
    ("FN (prédit baisse, hausse réelle)", "orange",    "Faux Négatif"),
]:
    sub = err_df[err_df["error_type"] == etype]["y_prob"]
    if len(sub):
        axes[0, 0].hist(sub, bins=bins, alpha=0.6, label=f"{label} (n={len(sub)})", color=color, density=True)
axes[0, 0].axvline(0.5, color="black", ls="--", lw=1)
axes[0, 0].set_xlabel("Probabilité prédite (P(hausse))")
axes[0, 0].set_ylabel("Densité")
axes[0, 0].set_title("Distribution de confiance par type d'erreur")
axes[0, 0].legend(fontsize=9)

err_df["correct_int"] = err_df["correct"].astype(int)
rolling_acc = err_df["correct_int"].rolling(30, min_periods=10).mean()
axes[0, 1].plot(rolling_acc, color="steelblue", lw=1.5, label="Accuracy glissante 30j")
axes[0, 1].axhline(err_df["correct_int"].mean(), color="black", ls="--", lw=1,
                    label=f"Moyenne ({err_df['correct_int'].mean():.1%})")
axes[0, 1].fill_between(rolling_acc.index, 0.4, rolling_acc, alpha=0.2, color="steelblue")
axes[0, 1].set_ylim([0.3, 0.9])
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Accuracy glissante (30j)")
axes[0, 1].set_title("Évolution temporelle des erreurs (test set)")
axes[0, 1].legend(fontsize=9)

if has_ret:
    errors_sub = err_df[err_df["error_type"] != "correct"]["btc_ret_real"].dropna()
    correct_sub = err_df[err_df["error_type"] == "correct"]["btc_ret_real"].dropna()
    axes[1, 0].boxplot(
        [correct_sub * 100, errors_sub * 100],
        labels=["Correct", "Erreur"],
        patch_artist=True,
        boxprops=dict(facecolor="lightblue"),
    )
    axes[1, 0].axhline(0, color="black", ls="--", lw=1)
    axes[1, 0].set_ylabel("Rendement BTC réel (%)")
    axes[1, 0].set_title("Amplitude du mouvement BTC lors des erreurs")
else:
    axes[1, 0].text(0.5, 0.5, "Données de rendement\nnon disponibles",
                    ha="center", va="center", transform=axes[1, 0].transAxes)

err_monthly = err_df.copy()
err_monthly["month"] = err_monthly.index.to_period("M").astype(str)
monthly_err_rate = 1 - err_monthly.groupby("month")["correct_int"].mean()
bars = axes[1, 1].bar(range(len(monthly_err_rate)), monthly_err_rate.values,
                       color=["tomato" if v > 0.5 else "steelblue" for v in monthly_err_rate.values])
axes[1, 1].axhline(monthly_err_rate.mean(), color="black", ls="--", lw=1,
                    label=f"Taux d'erreur moyen ({monthly_err_rate.mean():.1%})")
axes[1, 1].set_xticks(range(len(monthly_err_rate)))
axes[1, 1].set_xticklabels(monthly_err_rate.index, rotation=45, ha="right", fontsize=7)
axes[1, 1].set_ylabel("Taux d'erreur mensuel")
axes[1, 1].set_title("Taux d'erreur par mois (test set)")
axes[1, 1].legend(fontsize=9)
axes[1, 1].set_ylim([0, 1])

plt.suptitle(f"Analyse des erreurs — {best_name} (test set ≥ 2024)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("models/error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 4 — Feature Importance (SHAP)

SHAP (SHapley Additive exPlanations) quantifie la contribution de chaque feature à chaque prédiction.  
Pour les modèles DL (boîtes noires), on utilise `shap.DeepExplainer` ou `KernelExplainer` (plus lent mais universel).  
Pour XGBoost, on utilise `shap.TreeExplainer` (exact et rapide).

In [ ]:
try:
    import shap
    shap_available = True
    print(f"SHAP {shap.__version__} disponible.")
except ImportError:
    shap_available = False
    print("SHAP non installé. Installation : pip install shap")
    print("Les graphiques de feature importance seront basés sur les modèles sklearn/XGBoost natifs.")

feature_names = list(X_train.columns)

if shap_available and "XGBoost" in dl_results:
    xgb_model = dl_results["XGBoost"]["trainer"].model
    X_te_last = X_te_seq[:, -1, :]  # (n_test, n_features) — features du jour J-1

    explainer_xgb = shap.Explainer(xgb_model)
    shap_values_xgb = explainer_xgb(X_te_last)

    mean_abs_shap = np.abs(shap_values_xgb.values).mean(axis=0)
    if mean_abs_shap.shape[0] > len(feature_names):
        # Réorganiser : (n_features, seq_len) puis moyenner
        seq_len_used = mean_abs_shap.shape[0] // len(feature_names)
        mean_abs_shap = mean_abs_shap.reshape(seq_len_used, len(feature_names)).mean(axis=0)

    top_idx = np.argsort(mean_abs_shap)[-20:][::-1]
    top_features = [feature_names[i] for i in top_idx]
    top_values   = mean_abs_shap[top_idx]

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_features)))
    ax.barh(range(len(top_features)), top_values[::-1], color=colors)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features[::-1], fontsize=10)
    ax.set_xlabel("Importance SHAP moyenne |E[|φ(x)|]|")
    ax.set_title("Top 20 features — XGBoost (SHAP TreeExplainer, test set)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig("models/shap_xgboost.png", dpi=150, bbox_inches="tight")
    plt.show()

elif not shap_available:
    if "Random Forest" in sklearn_models_trained:
        rf_model = sklearn_models_trained["Random Forest"]
        importances = rf_model.feature_importances_
        top_idx = np.argsort(importances)[-20:][::-1]
        top_features = [feature_names[i] for i in top_idx]
        top_values   = importances[top_idx]

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(range(len(top_features)), top_values[::-1], color="steelblue")
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features[::-1], fontsize=10)
        ax.set_xlabel("Importance (Gini impurity decrease)")
        ax.set_title("Top 20 features — Random Forest (importance native)")
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig("models/feature_importance_rf.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Aucun modèle disponible pour la feature importance.")
